# Continue original TabPFN — unexecuted GPU continuation
Run after the original notebook in its original Colab session. No checkpoint/MLflow database was attached, so this notebook has not been executed here. It uses the original fitted model, original medians, original category domains, and original HH:MM-only time policy. It does not apply the revised Excel-time parser or refit on test.
Copy the food_delivery package to the working directory first.


In [ ]:
import copy, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import mlflow
from food_delivery.preprocessing import Preprocessor, FEATURES
from food_delivery.model import DeliveryModel
from food_delivery.train import metrics

required_state = ["model_5000", "train_medians", "time_medians", "X_train", "X_val", "val_df", "test_df", "feature_cols", "categorical_features"]
missing = [name for name in required_state if name not in globals()]
if missing:
    raise RuntimeError(f"Run original notebook first; missing session state: {missing}")
assert list(feature_cols) == FEATURES
config = {
    "schema_version": 1, "features": list(feature_cols),
    "medians": {**{c: float(train_medians[c]) for c in ["Delivery_person_Age", "Delivery_person_Ratings"]},
                **{c: float(time_medians[c]) for c in ["order_hour", "order_minute"]}},
    "categories": {c: X_train[c].cat.categories.tolist() for c in categorical_features},
    "time_mode": "legacy_hhmm_only", "origin": "frozen original notebook training state",
    "multiple_deliveries": "not a model feature; not required for raw inference",
}
frozen = Preprocessor(copy.deepcopy(config))
X_rebuilt = frozen.transform(val_df)
# Fixed category domains are intentional; compare values and feature order independently of categorical domain metadata.
pd.testing.assert_frame_equal(X_val.astype(object), X_rebuilt.astype(object), check_dtype=False, check_exact=False, rtol=1e-10, atol=1e-10)
np.testing.assert_allclose(model_5000.predict(X_val.head(20)), model_5000.predict(X_rebuilt.head(20)), rtol=1e-5, atol=1e-5)
Path("original_tabpfn_artifacts").mkdir(exist_ok=True)
frozen.save("original_tabpfn_artifacts/preprocessing_config.json")
print("Original training/inference feature and prediction consistency: PASSED")


## Frozen full test evaluation
Original model configuration was selected on validation. Evaluate every test row in batches. Do not tune hyperparameters or select another training subset after seeing these metrics.


In [ ]:
bundle = DeliveryModel(frozen, model_5000, {"name": "original_tabpfn_5000"})
start = time.perf_counter()
pred = np.concatenate([bundle.predict(test_df.iloc[i:i+100]) for i in range(0, len(test_df), 100)])
assert len(pred) == len(test_df)
result = metrics(test_df["Time_taken (min)"], pred)
result["prediction_seconds"] = time.perf_counter() - start
result["test_rows"] = len(test_df)
Path("original_tabpfn_artifacts/test_metrics.json").write_text(json.dumps(result, indent=2))
pd.DataFrame({"ID": test_df.ID, "actual": test_df["Time_taken (min)"], "prediction": pred}).to_csv("original_tabpfn_artifacts/test_predictions.csv", index=False)
print(result)
with mlflow.start_run(run_name="original_tabpfn_frozen_full_test"):
    mlflow.log_params({"model": "TabPFN original 5000", "n_estimators": 1, "fit_mode": "low_memory", "random_state": 42, "batch_size": 100})
    mlflow.log_metrics({f"test_{k}": float(v) for k,v in result.items()})
    mlflow.log_artifacts("original_tabpfn_artifacts")
    mlflow.set_tag("test_policy", "full chronological held-out test; no post-test tuning")


The MAE gate below is a safety check, not permission to tune on this test. Model size, raw-input MLflow reload and HTTP latency gates are still required before deployment.


In [ ]:
assert result["mae"] <= 5.0, "Test MAE gate failed; block promotion and investigate without tuning on this test"
print("Test quality gate: PASSED; serving gates remain required")
